In [ ]:
import ast
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parent.parent
RECIPE_FILE = Path("data/processed/Recipes_cleaned.csv")

df = pd.read_csv(RECIPE_FILE)

keywords = ["beef", "carrot", "cow pea", "pork", "milk"]

for _, row in df.iterrows():
    title = str(row.get("title_clean", row.get("title", "Unknown recipe")))
    raw_ingredients = row.get("ingredients_normalized", "")

    if pd.isna(raw_ingredients):
        continue

    try:
        ingredients = ast.literal_eval(str(raw_ingredients))
        if not isinstance(ingredients, list):
            ingredients = str(raw_ingredients).split("|")
    except (ValueError, SyntaxError):
        ingredients = str(raw_ingredients).split("|")

    ingredients = [str(item).strip() for item in ingredients]

    matched_keywords = [
        keyword
        for keyword in keywords
        if any(keyword in ingredient.lower() for ingredient in ingredients)
    ]

    if matched_keywords:
        print(f"\nRecipe: {title}")
        print(f"Keywords found: {', '.join(matched_keywords)}")
        print(f"Ingredients: {ingredients}")

In [ ]:

from pathlib import Path
import json
import re
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parent.parent

DATASET_PATH = Path("C:/Users/User/Documents/3RD YEAR/3.2/Artificial Intelligence/SemProject_MLEngine/data/raw/combined_recipes.csv")
OUTPUT_PATH = Path("C:/Users/User/Documents/3RD YEAR/3.2/Artificial Intelligence/SemProject_MLEngine/data/processed/Recipes_cleaned.csv")

INGREDIENT_ALIASES = {
    "cow milk": "milk",
    "cardamoms powder": "cardamom",
    "cardamoms": "cardamom",
    "spring onions": "spring onion",
    "green bananas": "green banana",
    "potatoes": "potato",
    "onions": "onion",
    "eggs": "egg",
}

SPECIAL_INGREDIENTS = {
    "wheat flour, refined": ["wheat flour"],
    "water, vanilla essence, clear": ["water", "vanilla essence"],
    "yeast, cooking oil, sugar": ["yeast", "cooking oil", "sugar"],
    "coriander, fresh": ["coriander"],
    "sugar, white": ["sugar"],
    "food colour powder, egg yellow": ["food colour powder"],
    "njahi, dolichos lablab": ["njahi"],
    "black beans, dolichos": ["njahi"],
    "rind from lemons sugar": ["lemon rind", "sugar"],
}


def load_dataset():
    if not DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found: {DATASET_PATH}"
        )

    df = pd.read_csv(DATASET_PATH)

    required_columns = {"title", "ingredients"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if df.empty:
        raise ValueError("The dataset contains no recipes.")

    return df


def normalize_ingredient(ingredient):
    ingredient = str(ingredient).lower().strip()
    ingredient = re.sub(r"\s+", " ", ingredient)
    ingredient = ingredient.strip(" ,;.")

    return INGREDIENT_ALIASES.get(
        ingredient,
        ingredient
    )


def parse_ingredients(raw_ingredients):
    if pd.isna(raw_ingredients):
        return []

    raw = str(raw_ingredients).strip()

    if not raw:
        return []

    ingredients = []

    if "|" in raw:
        parts = raw.split("|")

    else:
        parts = []

        for line in raw.splitlines():
            line = re.sub(r"\s+", " ", line).strip()
            line = line.strip(" ,;.")

            if not line:
                continue

            if line in SPECIAL_INGREDIENTS:
                parts.extend(SPECIAL_INGREDIENTS[line])
            else:
                parts.append(line)

    for part in parts:
        ingredient = normalize_ingredient(part)

        if ingredient:
            ingredients.append(ingredient)

    unique_ingredients = []
    seen = set()

    for ingredient in ingredients:
        if ingredient not in seen:
            unique_ingredients.append(ingredient)
            seen.add(ingredient)

    return unique_ingredients


def normalize_title(title):
    title = str(title).lower().strip()
    title = re.sub(r"\s+", " ", title)
    return title


def preprocess_dataset(df):
    df = df.copy()

    df["title_clean"] = (
        df["title"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df["ingredients_list"] = df["ingredients"].apply(
        parse_ingredients
    )

    df["ingredients_normalized"] = df[
        "ingredients_list"
    ].apply(
        lambda items: json.dumps(
            items,
            ensure_ascii=False
        )
    )

    df["title_key"] = df["title_clean"].apply(
        normalize_title
    )

    df["ingredient_set_key"] = df[
        "ingredients_list"
    ].apply(
        lambda items: json.dumps(
            sorted(set(items)),
            ensure_ascii=False
        )
    )

    return df


def remove_empty_recipes(df):
    empty_mask = df["ingredients_list"].apply(
        lambda items: len(items) == 0
    )

    removed_count = int(empty_mask.sum())

    cleaned_df = df.loc[~empty_mask].copy()

    print(
        f"Recipes removed because they have no ingredients: "
        f"{removed_count}"
    )

    return cleaned_df


def remove_exact_duplicate_recipes(df):
    duplicate_mask = df.duplicated(
        subset=["title_key", "ingredient_set_key"],
        keep="first"
    )

    removed_count = int(duplicate_mask.sum())

    cleaned_df = df.loc[~duplicate_mask].copy()

    print(
        f"Exact duplicate recipes removed: {removed_count}"
    )

    return cleaned_df


def report_results(df, original_count):
    df["title_key"] = (
    df["title_clean"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)
    print("\n Cleaning Results ")

    print(f"Original recipe count: {original_count}")
    print(f"Remaining recipe count: {len(df)}")

    print(
        "Recipes with no parsed ingredients:",
        int(df["ingredients_list"].apply(len).eq(0).sum())
    )

    total_ingredient_occurrences = sum(
        len(items) for items in df["ingredients_list"]
    )

    unique_ingredient_occurrences = sum(
        len(set(items)) for items in df["ingredients_list"]
    )

    print(
        "Ingredient occurrences after deduplication:",
        total_ingredient_occurrences
    )

    print(
        "Unique ingredient occurrences within recipes:",
        unique_ingredient_occurrences
    )

    duplicate_titles = df[
        df["title_key"].duplicated(keep=False)
    ].sort_values("title_key")

    print(
        "Rows with titles still duplicated:",
        len(duplicate_titles)
    )

    if not duplicate_titles.empty:
        print("\nDuplicate titles requiring review:")

        for _, row in duplicate_titles.iterrows():
            print(f"\nRecipe: {row['title_clean']}")
            print(f"Ingredients: {row['ingredients_list']}")

    all_ingredients = sorted({
        ingredient
        for items in df["ingredients_list"]
        for ingredient in items
    })

    print(
        "\nNumber of distinct normalized ingredient names:",
        len(all_ingredients)
    )

    print("\nExamples of cleaned recipes")

    for _, row in df.head(5).iterrows():
        print(f"\nRecipe: {row['title_clean']}")
        print(f"Ingredients: {row['ingredients_list']}")

    print("\n First 30 ingredients")

    for ingredient in all_ingredients[:30]:
        print(f"- {ingredient}")


def main():
    df = load_dataset()
    original_count = len(df)

    df = preprocess_dataset(df)

    df = remove_empty_recipes(df)

    df = remove_exact_duplicate_recipes(df)

    df = df.drop(
        columns=["title_key", "ingredient_set_key"]
    )

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(OUTPUT_PATH, index=False)

    report_results(df, original_count)

    print(f"\nCleaned dataset saved to:\n{OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:

import json
from pathlib import Path
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = Path("data/processed/Training_data")
MODEL_DIR = Path("models")

FEATURE_COLUMNS = [
    "jaccard_similarity",
    "user_coverage",
    "recipe_coverage",
    "matched_ingredients",
    "missing_ingredients",
    "user_ingredient_count",
    "recipe_ingredient_count",
]

TARGET_COLUMN = "label"


def load_dataset(filename):
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    return pd.read_csv(path)


def validate_dataset(df, name):
    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {missing_columns}"
        )

    if df[required_columns].isna().any().any():
        raise ValueError(
            f"{name} contains missing feature or label values."
        )

    labels = set(df[TARGET_COLUMN].unique())

    if labels != {0, 1}:
        raise ValueError(
            f"{name} must contain both labels 0 and 1. "
            f"Found: {labels}"
        )


def evaluate_model(model, df, dataset_name):
    X = df[FEATURE_COLUMNS]
    y = df[TARGET_COLUMN].astype(int)

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y, predictions)),
        "precision": float(
            precision_score(y, predictions, zero_division=0)
        ),
        "recall": float(
            recall_score(y, predictions, zero_division=0)
        ),
        "f1_score": float(
            f1_score(y, predictions, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y, probabilities)),
    }

    print(f"\n{dataset_name.upper()} Results")

    for metric, value in metrics.items():
        print(f"{metric.replace('_', ' ').title()}: {value:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y,
            predictions,
            target_names=["Negative", "Positive"],
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            y,
            predictions,
            labels=[0, 1],
        )
    )

    return metrics


def main():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    train_df = load_dataset("training_examples.csv")
    validation_df = load_dataset("validation_examples.csv")
    test_df = load_dataset("test_examples.csv")

    validate_dataset(train_df, "Training dataset")
    validate_dataset(validation_df, "Validation dataset")
    validate_dataset(test_df, "Test dataset")

    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMN].astype(int)

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )

    print("Training Logistic Regression model...")

    model.fit(X_train, y_train)

    print("Training completed.")

    validation_metrics = evaluate_model(
        model,
        validation_df,
        "Validation",
    )

    test_metrics = evaluate_model(
        model,
        test_df,
        "Test",
    )

    model_path = MODEL_DIR / "recipe_match_model.joblib"

    joblib.dump(model, model_path)

    metrics_path = MODEL_DIR / "evaluation_metrics.json"

    with open(metrics_path, "w", encoding="utf-8") as file:
        json.dump(
            {
                "features": FEATURE_COLUMNS,
                "validation": validation_metrics,
                "test": test_metrics,
                "training_labels_are_synthetic": True,
            },
            file,
            indent=4,
        )

    print("\nSaved files")
    print(f"Trained model: {model_path}")
    print(f"Evaluation metrics: {metrics_path}")


if __name__ == "__main__":
    main()

In [ ]:

import ast
from importlib.resources import path
import json
import random
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


PROJECT_ROOT = Path(__file__).resolve().parent.parent
INPUT_FILE = Path("data/processed/Recipes_cleaned.csv")
OUTPUT_FILE = Path("data/processed/Training_data")

RANDOM_SEED = 42
NEGATIVE_EXAMPLES_PER_RECIPE = 3


def parse_ingredients(value):
    """Convert a stored ingredient list into a normalized set."""

    if isinstance(value, list):
        ingredients = value

    elif pd.isna(value):
        return set()

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return set()

        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                ingredients = parsed
            else:
                ingredients = value.split("|")

        except (ValueError, SyntaxError):
            ingredients = value.split("|")

    else:
        return set()

    return {
        str(ingredient).strip().lower()
        for ingredient in ingredients
        if str(ingredient).strip()
    }


def calculate_features(user_ingredients, recipe_ingredients):
    """Calculate ingredient similarity features for a user–recipe pair."""

    matched = user_ingredients.intersection(recipe_ingredients)
    union = user_ingredients.union(recipe_ingredients)

    user_count = len(user_ingredients)
    recipe_count = len(recipe_ingredients)
    matched_count = len(matched)

    jaccard_similarity = (
        len(matched) / len(union)
        if union
        else 0.0
    )

    user_coverage = (
        matched_count / user_count
        if user_count
        else 0.0
    )

    recipe_coverage = (
        matched_count / recipe_count
        if recipe_count
        else 0.0
    )

    missing_ingredients = recipe_count - matched_count

    return {
        "jaccard_similarity": jaccard_similarity,
        "user_coverage": user_coverage,
        "recipe_coverage": recipe_coverage,
        "matched_ingredients": matched_count,
        "missing_ingredients": missing_ingredients,
        "user_ingredient_count": user_count,
        "recipe_ingredient_count": recipe_count,
    }


def create_user_ingredients(recipe_ingredients, rng):
    """Simulate ingredients available to a user."""

    ingredients = sorted(recipe_ingredients)

    if not ingredients:
        return set()

    if len(ingredients) == 1:
        return set(ingredients)

    minimum = max(1, int(len(ingredients) * 0.4))
    maximum = max(minimum, int(len(ingredients) * 0.8))
    maximum = min(maximum, len(ingredients) - 1)

    selected_count = rng.randint(minimum, maximum)

    return set(rng.sample(ingredients, selected_count))


def generate_examples(recipes, split_name, rng):
    """Generate positive and negative training examples."""

    examples = []

    recipes = recipes.reset_index(drop=True)

    ingredient_sets = recipes["ingredient_set"].tolist()
    titles = recipes["title_clean"].astype(str).tolist()

    for anchor_index, anchor_ingredients in enumerate(ingredient_sets):

        if not anchor_ingredients:
            continue

        user_ingredients = create_user_ingredients(
            anchor_ingredients,
            rng,
        )

        candidate_indices = [
            index
            for index in range(len(recipes))
            if index != anchor_index
        ]

        if not candidate_indices:
            continue

        negative_count = min(
            NEGATIVE_EXAMPLES_PER_RECIPE,
            len(candidate_indices),
        )

        negative_indices = rng.sample(
            candidate_indices,
            negative_count,
        )

        candidates = [(anchor_index, 1)]

        candidates.extend(
            (index, 0)
            for index in negative_indices
        )

        for candidate_index, label in candidates:

            candidate_ingredients = ingredient_sets[candidate_index]

            features = calculate_features(
                user_ingredients,
                candidate_ingredients,
            )

            example = {
                "split": split_name,
                "user_ingredients": json.dumps(
                    sorted(user_ingredients)
                ),
                "candidate_title": titles[candidate_index],
                "label": label,
                **features,
            }

            examples.append(example)

    return pd.DataFrame(examples)


def split_recipes(df):
    """Split recipes by title group to reduce data leakage."""

    groups = (
        df["title_clean"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    first_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=RANDOM_SEED,
    )

    train_indices, remaining_indices = next(
        first_split.split(df, groups=groups)
    )

    train_recipes = df.iloc[train_indices].copy()
    remaining_recipes = df.iloc[remaining_indices].copy()

    remaining_groups = groups.iloc[remaining_indices]

    second_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=RANDOM_SEED,
    )

    validation_indices, test_indices = next(
        second_split.split(
            remaining_recipes,
            groups=remaining_groups,
        )
    )

    validation_recipes = remaining_recipes.iloc[
        validation_indices
    ].copy()

    test_recipes = remaining_recipes.iloc[
        test_indices
    ].copy()

    return train_recipes, validation_recipes, test_recipes


def main():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Cleaned dataset not found: {INPUT_FILE}\n"
            "Run src/preprocessing.py first."
        )

    OUTPUT_FILE.mkdir(parents=True, exist_ok=True)

    rng = random.Random(RANDOM_SEED)

    df = pd.read_csv(INPUT_FILE)

    if "title_clean" not in df.columns:
        raise ValueError(
            "The cleaned dataset must contain 'title_clean'."
        )

    ingredient_column = (
        "ingredients_normalized"
        if "ingredients_normalized" in df.columns
        else "ingredients_list"
    )

    if ingredient_column not in df.columns:
        raise ValueError(
            "No ingredient column was found. Expected "
            "'ingredients_normalized' or 'ingredients_list'. "
            f"Available columns: {df.columns.tolist()}"
        )

    df["ingredient_set"] = df[ingredient_column].apply(
        parse_ingredients
    )

    original_count = len(df)

    df = df[
        df["ingredient_set"].map(len) > 0
    ].copy()

    print("========== DATA PREPARATION ==========")
    print(f"Recipes loaded: {original_count}")
    print(f"Recipes with usable ingredients: {len(df)}")
    print(f"Ingredient column used: {ingredient_column}")

    train_recipes, validation_recipes, test_recipes = (
        split_recipes(df)
    )

    print("\n========== RECIPE SPLITS ==========")
    print(f"Training recipes: {len(train_recipes)}")
    print(f"Validation recipes: {len(validation_recipes)}")
    print(f"Test recipes: {len(test_recipes)}")

    train_examples = generate_examples(
        train_recipes,
        "train",
        rng,
    )

    validation_examples = generate_examples(
        validation_recipes,
        "validation",
        rng,
    )

    test_examples = generate_examples(
        test_recipes,
        "test",
        rng,
    )

    datasets = {
        "training_examples.csv": train_examples,
        "validation_examples.csv": validation_examples,
        "test_examples.csv": test_examples,
    }

    for filename, dataset in datasets.items():
        output_path = OUTPUT_FILE / filename
        dataset.to_csv(output_path, index=False)

        print(f"\nSaved: {output_path}")
        print(f"Examples: {len(dataset)}")

        if not dataset.empty:
            print(
                "Label distribution:",
                dataset["label"].value_counts().sort_index().to_dict(),
            )

    print("\n========== FEATURE COLUMNS ==========")

    if not train_examples.empty:
        print(
            train_examples[
                [
                    "jaccard_similarity",
                    "user_coverage",
                    "recipe_coverage",
                    "matched_ingredients",
                    "missing_ingredients",
                    "user_ingredient_count",
                    "recipe_ingredient_count",
                    "label",
                ]
            ].head()
        )

    print("\nTraining-data generation completed.")


if __name__ == "__main__":
    main()


In [ ]:
#train model 1
import json
from pathlib import Path
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = Path("data/processed/Training_data")
MODEL_DIR = Path("models")

FEATURE_COLUMNS = [
    "jaccard_similarity",
    "user_coverage",
    "recipe_coverage",
    "matched_ingredients",
    "missing_ingredients",
    "user_ingredient_count",
    "recipe_ingredient_count",
]

TARGET_COLUMN = "label"


def load_dataset(filename):
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    return pd.read_csv(path)


def validate_dataset(df, name):
    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {missing_columns}"
        )

    if df[required_columns].isna().any().any():
        raise ValueError(
            f"{name} contains missing feature or label values."
        )

    labels = set(df[TARGET_COLUMN].unique())

    if labels != {0, 1}:
        raise ValueError(
            f"{name} must contain both labels 0 and 1. "
            f"Found: {labels}"
        )


def evaluate_model(model, df, dataset_name):
    X = df[FEATURE_COLUMNS]
    y = df[TARGET_COLUMN].astype(int)

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y, predictions)),
        "precision": float(
            precision_score(y, predictions, zero_division=0)
        ),
        "recall": float(
            recall_score(y, predictions, zero_division=0)
        ),
        "f1_score": float(
            f1_score(y, predictions, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y, probabilities)),
    }

    print(f"\n{dataset_name.upper()} Results")

    for metric, value in metrics.items():
        print(f"{metric.replace('_', ' ').title()}: {value:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y,
            predictions,
            target_names=["Negative", "Positive"],
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            y,
            predictions,
            labels=[0, 1],
        )
    )

    return metrics


def main():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    train_df = load_dataset("training_examples.csv")
    validation_df = load_dataset("validation_examples.csv")
    test_df = load_dataset("test_examples.csv")

    validate_dataset(train_df, "Training dataset")
    validate_dataset(validation_df, "Validation dataset")
    validate_dataset(test_df, "Test dataset")

    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMN].astype(int)

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )

    print("Training Logistic Regression model...")

    model.fit(X_train, y_train)

    print("Training completed.")

    validation_metrics = evaluate_model(
        model,
        validation_df,
        "Validation",
    )

    test_metrics = evaluate_model(
        model,
        test_df,
        "Test",
    )

    model_path = MODEL_DIR / "recipe_match_model.joblib"

    joblib.dump(model, model_path)

    metrics_path = MODEL_DIR / "evaluation_metrics.json"

    with open(metrics_path, "w", encoding="utf-8") as file:
        json.dump(
            {
                "features": FEATURE_COLUMNS,
                "validation": validation_metrics,
                "test": test_metrics,
                "training_labels_are_synthetic": True,
            },
            file,
            indent=4,
        )

    print("\nSaved files")
    print(f"Trained model: {model_path}")
    print(f"Evaluation metrics: {metrics_path}")


if __name__ == "__main__":
    main()

In [ ]:
#recommend.py

import ast
import re
from pathlib import Path
import numpy as np
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parent.parent

RECIPE_FILE = Path("data/processed/Recipes_cleaned.csv")

TOP_N = 10

# Semantic matching is enabled, but should be evaluated carefully.
ENABLE_SEMANTIC_MATCHING = True

# Conservative starting threshold, not a guarantee of correctness.
SEMANTIC_THRESHOLD = 0.62

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# INGREDIENT NORMALISATION
PREPARATION_WORDS = {
    "fresh",
    "ripe",
    "raw",
    "cooked",
    "boiled",
    "chopped",
    "diced",
    "sliced",
    "peeled",
    "grated",
    "crushed",
    "frozen",
}

NON_PLURAL_S_WORDS = {
    "asparagus",
    "couscous",
    "hummus",
    "molasses",
}

# Cache embeddings to avoid repeatedly encoding the same ingredient.
EMBEDDING_CACHE = {}

SEMANTIC_MODEL = None


def parse_ingredients(value):
    """Convert the dataset ingredient field into a list."""
    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]

    except (ValueError, SyntaxError):
        pass

    if "|" in text:
        return [
            item.strip()
            for item in text.split("|")
            if item.strip()
        ]

    return [text]


def singularize_word(word):
    """Handle common plural ingredient names."""
    if word in NON_PLURAL_S_WORDS:
        return word

    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"

    if len(word) > 4 and word.endswith("oes"):
        return word[:-2]

    if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
        return word[:-1]

    return word


def clean_ingredient_text(ingredient):
    """
    Remove parenthetical descriptions and preparation words
    while retaining the meaningful ingredient name.
    """
    ingredient = str(ingredient).lower().strip()

    ingredient = re.sub(r"\([^)]*\)", "", ingredient)

    ingredient = ingredient.replace("_", " ")
    ingredient = ingredient.replace("-", " ")

    ingredient = re.sub(r"[^a-z0-9\s]", " ", ingredient)
    ingredient = re.sub(r"\s+", " ", ingredient).strip()

    words = ingredient.split()

    while words and words[0] in PREPARATION_WORDS:
        words.pop(0)

    return " ".join(words)


def normalize_ingredient(ingredient):
    """Create a canonical name for exact ingredient matching."""
    cleaned = clean_ingredient_text(ingredient)

    if not cleaned:
        return ""

    words = cleaned.split()
    words[-1] = singularize_word(words[-1])

    return " ".join(words)


def normalize_ingredient_list(ingredients):
    """Return unique normalised ingredient names."""
    return {
        normalize_ingredient(item)
        for item in ingredients
        if normalize_ingredient(item)
    }



# SEMANTIC MATCHING


def load_semantic_model():
    """Load the pretrained embedding model when required."""
    global SEMANTIC_MODEL

    if SEMANTIC_MODEL is not None:
        return SEMANTIC_MODEL

    try:
        from sentence_transformers import SentenceTransformer

        SEMANTIC_MODEL = SentenceTransformer(MODEL_NAME)

        return SEMANTIC_MODEL

    except ImportError as error:
        raise RuntimeError(
            "Semantic matching requires sentence-transformers. "
        ) from error


def get_embeddings(ingredient_names):
    """
    Encode ingredient names and cache their embeddings.
    Embeddings are normalised so their dot product gives
    cosine similarity.
    """
    unique_names = list(dict.fromkeys(
        name for name in ingredient_names if name
    ))

    missing_names = [
        name for name in unique_names
        if name not in EMBEDDING_CACHE
    ]

    if missing_names:
        model = load_semantic_model()

        embeddings = model.encode(
            missing_names,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        for name, embedding in zip(missing_names, embeddings):
            EMBEDDING_CACHE[name] = embedding

    return {
        name: EMBEDDING_CACHE[name]
        for name in unique_names
    }


def is_conflicting_milk_match(first, second):
    
    plant_milks = {
        "coconut milk",
        "almond milk",
        "oat milk",
        "soy milk",
        "soya milk",
        "rice milk",
    }

    first = normalize_ingredient(first)
    second = normalize_ingredient(second)

    if first == second:
        return False

    return (
        (first == "milk" and second in plant_milks)
        or (second == "milk" and first in plant_milks)
    )


def semantic_similarity(first, second, embeddings):
    """Calculate cosine similarity between two ingredient names."""
    first_text = clean_ingredient_text(first)
    second_text = clean_ingredient_text(second)

    if not first_text or not second_text:
        return 0.0

    first_embedding = embeddings.get(first_text)
    second_embedding = embeddings.get(second_text)

    if first_embedding is None or second_embedding is None:
        return 0.0

    return float(np.dot(first_embedding, second_embedding))



def match_ingredients(user_ingredients, recipe_ingredients):
    """
    Match ingredients in two stages.

    Stage 1: exact normalised matches.
    Stage 2: semantic matches for unmatched ingredients.

    A user ingredient and a recipe ingredient can each be
    matched only once.
    """
    user_items = {}

    for item in user_ingredients:
        normalized = normalize_ingredient(item)

        if normalized:
            user_items.setdefault(normalized, item)

    recipe_items = {}

    for item in recipe_ingredients:
        normalized = normalize_ingredient(item)

        if normalized:
            recipe_items.setdefault(normalized, item)

    unmatched_users = set(user_items)
    unmatched_recipes = set(recipe_items)

    matched_pairs = []

    # Stage 1: exact matches after normalisation.
    exact_matches = sorted(
        unmatched_users & unmatched_recipes
    )

    for name in exact_matches:
        matched_pairs.append({
            "user_ingredient": user_items[name],
            "recipe_ingredient": recipe_items[name],
            "similarity": 1.0,
            "match_type": "Exact",
        })

        unmatched_users.remove(name)
        unmatched_recipes.remove(name)

    # Stage 2: semantic matching for remaining ingredients.
    if ENABLE_SEMANTIC_MATCHING and unmatched_users and unmatched_recipes:
        user_candidates = [
            user_items[name]
            for name in unmatched_users
        ]

        recipe_candidates = [
            recipe_items[name]
            for name in unmatched_recipes
        ]

        candidate_texts = [
            clean_ingredient_text(item)
            for item in user_candidates + recipe_candidates
        ]

        embeddings = get_embeddings(candidate_texts)

        possible_matches = []

        for user_name in unmatched_users:
            user_original = user_items[user_name]

            for recipe_name in unmatched_recipes:
                recipe_original = recipe_items[recipe_name]

                # Protect against known ingredient conflicts.
                if is_conflicting_milk_match(
                    user_original,
                    recipe_original,
                ):
                    continue

                score = semantic_similarity(
                    user_original,
                    recipe_original,
                    embeddings,
                )

                if score >= SEMANTIC_THRESHOLD:
                    possible_matches.append(
                        (
                            score,
                            user_name,
                            recipe_name,
                        )
                    )

        # Consider the strongest matches first.
        possible_matches.sort(reverse=True)

        for score, user_name, recipe_name in possible_matches:
            if (
                user_name not in unmatched_users
                or recipe_name not in unmatched_recipes
            ):
                continue

            matched_pairs.append({
                "user_ingredient": user_items[user_name],
                "recipe_ingredient": recipe_items[recipe_name],
                "similarity": score,
                "match_type": "Semantic",
            })

            unmatched_users.remove(user_name)
            unmatched_recipes.remove(recipe_name)

    return matched_pairs


# ============================================================
# RECIPE SCORING AND RANKING
# ============================================================

def calculate_features(user_ingredients, recipe_ingredients):
    """Calculate ingredient overlap and recommendation scores."""
    matched_pairs = match_ingredients(
        user_ingredients,
        recipe_ingredients,
    )

    user_set = normalize_ingredient_list(user_ingredients)
    recipe_set = normalize_ingredient_list(recipe_ingredients)

    matched_count = len(matched_pairs)

    user_coverage = (
        matched_count / len(user_set)
        if user_set else 0.0
    )

    recipe_coverage = (
        matched_count / len(recipe_set)
        if recipe_set else 0.0
    )

    # Use the number of matched pairs and the normalised sets.
    matched_user_names = {
        normalize_ingredient(pair["user_ingredient"])
        for pair in matched_pairs
    }

    matched_recipe_names = {
        normalize_ingredient(pair["recipe_ingredient"])
        for pair in matched_pairs
    }

    union_count = (
        len(user_set)
        + len(recipe_set)
        - len(matched_user_names & matched_recipe_names)
    )

    # For semantic matches, the two normalised names may differ.
    # Compute a practical overlap score from the matched count.
    union_count = max(
        len(user_set) + len(recipe_set) - matched_count,
        0,
    )

    jaccard_similarity = (
        matched_count / union_count
        if union_count else 0.0
    )

    exact_count = sum(
        pair["match_type"] == "Exact"
        for pair in matched_pairs
    )

    semantic_count = sum(
        pair["match_type"] == "Semantic"
        for pair in matched_pairs
    )

    semantic_scores = [
        pair["similarity"]
        for pair in matched_pairs
        if pair["match_type"] == "Semantic"
    ]

    average_semantic_similarity = (
        sum(semantic_scores) / len(semantic_scores)
        if semantic_scores else 1.0
    )

    return {
        "matched_pairs": matched_pairs,
        "matched_count": matched_count,
        "exact_count": exact_count,
        "semantic_count": semantic_count,
        "average_semantic_similarity": average_semantic_similarity,
        "user_coverage": user_coverage,
        "recipe_coverage": recipe_coverage,
        "jaccard_similarity": jaccard_similarity,
    }


def load_recipes():
    """Load recipes from the cleaned CSV dataset."""
    if not RECIPE_FILE.exists():
        raise FileNotFoundError(
            f"Recipe dataset not found: {RECIPE_FILE}"
        )

    df = pd.read_csv(RECIPE_FILE)

    if "ingredients_normalized" not in df.columns:
        raise ValueError(
            "Missing 'ingredients_normalized' column in dataset."
        )

    if "title_clean" in df.columns:
        title_column = "title_clean"
    elif "title" in df.columns:
        title_column = "title"
    else:
        raise ValueError(
            "The dataset does not contain a recipe title column."
        )

    recipes = []

    for _, row in df.iterrows():
        ingredients = parse_ingredients(
            row["ingredients_normalized"]
        )

        if not normalize_ingredient_list(ingredients):
            continue

        title = str(row[title_column]).strip()

        if not title or title.lower() == "nan":
            title = "Untitled Recipe"

        recipes.append({
            "title": title,
            "ingredients": ingredients,
        })

    return recipes


def recommend_recipes(user_ingredients, recipes, top_n=TOP_N):
    """Recommend recipes with the strongest ingredient overlap."""
    if not normalize_ingredient_list(user_ingredients):
        return []

    recommendations = []

    for recipe in recipes:
        features = calculate_features(
            user_ingredients,
            recipe["ingredients"],
        )

        if features["matched_count"] == 0:
            continue

        recommendations.append({
            **recipe,
            **features,
        })

    # More matched ingredients always take priority.
    # Exact matches take priority over semantic matches on ties.
    # Similarity and coverage break remaining ties.
    recommendations.sort(
        key=lambda recipe: (
            recipe["matched_count"],
            recipe["exact_count"],
            recipe["average_semantic_similarity"],
            recipe["user_coverage"],
            recipe["jaccard_similarity"],
            recipe["recipe_coverage"],
        ),
        reverse=True,
    )

    return recommendations[:top_n]



def main():
    print("=" * 50)
    print("          RECIPE RECOMMENDATION SYSTEM")
    print("=" * 50)

    user_input = input(
        "\nEnter ingredients separated by commas: "
    ).strip()

    if not user_input:
        print("Please enter at least one ingredient.")
        return

    user_ingredients = [
        item.strip()
        for item in user_input.split(",")
        if item.strip()
    ]

    normalized_user = normalize_ingredient_list(
        user_ingredients
    )

    print(f"\nIngredients entered: {len(normalized_user)}")
    print("Ingredients:", ", ".join(sorted(normalized_user)))

    try:
        recipes = load_recipes()

        recommendations = recommend_recipes(
            user_ingredients,
            recipes,
        )

    except (FileNotFoundError, ValueError, RuntimeError) as error:
        print(f"\nError: {error}")
        return

    print("\n========== TOP RECOMMENDED RECIPES ==========")

    if not recommendations:
        print("No matching recipes were found.")
        print("Try entering additional ingredients.")
        return

    for index, recipe in enumerate(recommendations, start=1):
        print(f"\n{index}. {recipe['title']}")

        print(
            f"   Matched ingredients: "
            f"{recipe['matched_count']} of {len(normalized_user)}"
        )

        print(
            f"   User ingredient coverage: "
            f"{recipe['user_coverage']:.1%}"
        )

        print(
            f"   Recipe ingredient coverage: "
            f"{recipe['recipe_coverage']:.1%}"
        )

        print(
            f"   Jaccard similarity: "
            f"{recipe['jaccard_similarity']:.1%}"
        )

        print("   Matched ingredient details:")

        for pair in recipe["matched_pairs"]:
            user_item = pair["user_ingredient"]
            recipe_item = pair["recipe_ingredient"]
            match_type = pair["match_type"]

            if match_type == "Semantic":
                print(
                    f"      - {user_item} -> {recipe_item} "
                    f"(semantic similarity: {pair['similarity']:.3f})"
                )
            else:
                print(
                    f"      - {user_item} "
                    f"(dataset: {recipe_item}; exact match)"
                )


if __name__ == "__main__":
    main()


In [ ]:
#Generate embeddings

import re
import random
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer



BASE_DIR = Path(__file__).resolve().parent.parent

EMBEDDINGS_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_embeddings.npz"
)

EXISTING_REVIEW_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_pair_review.csv"
)

MANUAL_LABELS_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_matching_tests.csv"
)

OUTPUT_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_pair_candidates_200.csv"
)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TARGET_PAIRS = 200
ALIAS_TARGET = 50
TOP_NEIGHBOURS = 20
RANDOM_SEED = 42


# --------------------------------------------------
# 2. Load existing embeddings
# --------------------------------------------------

def load_embeddings():

    data = np.load(EMBEDDINGS_FILE)

    names = data["ingredients"]
    vectors = data["embeddings"]

    lookup = {
        str(name).lower().strip(): vector
        for name, vector in zip(names, vectors)
    }

    print(f"Loaded {len(lookup)} ingredient embeddings.")

    return lookup


# --------------------------------------------------
# 3. Load existing pairs to avoid duplicates
# --------------------------------------------------

def get_existing_pairs():

    existing_pairs = set()

    files = [
        EXISTING_REVIEW_FILE,
        MANUAL_LABELS_FILE
    ]

    for file_path in files:

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        for _, row in df.iterrows():

            a = str(row["ingredient_a"]).lower().strip()
            b = str(row["ingredient_b"]).lower().strip()

            if a and b and a != "nan" and b != "nan":
                existing_pairs.add(tuple(sorted((a, b))))

    print(f"Found {len(existing_pairs)} previously used pairs.")

    return existing_pairs


# --------------------------------------------------
# 4. Generate candidates from parenthetical names
# --------------------------------------------------

def generate_alias_candidates(ingredient_names):

    candidates = {}

    for name in ingredient_names:

        match = re.fullmatch(r"(.+?)\s*\(([^()]+)\)", name)

        if not match:
            continue

        main_name = match.group(1).strip()
        alias_name = match.group(2).strip()

        if not main_name or not alias_name:
            continue

        if main_name == alias_name:
            continue

        pair = tuple(sorted((main_name, alias_name)))

        candidates[pair] = "Parenthetical name candidate"

    return candidates


# --------------------------------------------------
# 5. Generate semantic-neighbour candidates
# --------------------------------------------------

def generate_semantic_candidates(names, vectors):

    similarity_matrix = np.dot(vectors, vectors.T)

    candidates = {}

    for index, name in enumerate(names):

        similarities = similarity_matrix[index]

        # Find the nearest ingredients, excluding itself.
        neighbour_indices = np.argsort(similarities)[
            -(TOP_NEIGHBOURS + 1):-1
        ]

        for neighbour_index in neighbour_indices:

            other_name = names[neighbour_index]

            if name == other_name:
                continue

            similarity = float(similarities[neighbour_index])

            if similarity < 0.30:
                continue

            pair = tuple(sorted((name, other_name)))

            candidates[pair] = similarity

    return candidates


# --------------------------------------------------
# 6. Calculate alias-pair similarity
# --------------------------------------------------

def calculate_alias_similarities(alias_candidates, lookup, model):

    missing_names = set()

    for pair in alias_candidates:

        for name in pair:

            if name not in lookup:
                missing_names.add(name)

    if missing_names:

        missing_names = sorted(missing_names)

        new_vectors = model.encode(
            missing_names,
            normalize_embeddings=True
        )

        for name, vector in zip(missing_names, new_vectors):
            lookup[name] = vector

    results = {}

    for pair, reason in alias_candidates.items():

        a, b = pair

        similarity = float(np.dot(lookup[a], lookup[b]))

        results[pair] = {
            "cosine_similarity": similarity,
            "candidate_source": reason
        }

    return results


# --------------------------------------------------
# 7. Select exactly 200 new pairs
# --------------------------------------------------

def select_candidates(alias_candidates, semantic_candidates, existing):

    random.seed(RANDOM_SEED)

    alias_available = [
        pair for pair in alias_candidates
        if pair not in existing
    ]

    semantic_available = [
        pair for pair in semantic_candidates
        if pair not in existing
    ]

    random.shuffle(alias_available)
    random.shuffle(semantic_available)

    selected = {}

    # Prioritize parenthetical-name candidates.
    alias_count = min(ALIAS_TARGET, len(alias_available))

    for pair in alias_available[:alias_count]:

        selected[pair] = {
            "candidate_source": "Parenthetical name candidate"
        }

    # Fill the remaining spaces with semantic neighbours.
    remaining = TARGET_PAIRS - len(selected)

    for pair in semantic_available:

        if remaining == 0:
            break

        if pair not in selected:

            selected[pair] = {
                "candidate_source": "Semantic neighbour candidate"
            }

            remaining -= 1

    # If semantic candidates overlap with alias candidates,
    # or there are not enough, try additional alias candidates.
    if remaining > 0:

        for pair in alias_available[alias_count:]:

            if remaining == 0:
                break

            if pair not in selected:

                selected[pair] = {
                    "candidate_source": "Parenthetical name candidate"
                }

                remaining -= 1

    if len(selected) < TARGET_PAIRS:

        raise ValueError(
            f"Only {len(selected)} new pairs were found. "
            "More candidates are needed."
        )

    return selected




def main():

    random.seed(RANDOM_SEED)

    lookup = load_embeddings()

    existing_pairs = get_existing_pairs()

    names = list(lookup.keys())

    vectors = np.array([
        lookup[name] for name in names
    ])

    print("\nGenerating parenthetical-name candidates...")

    alias_candidates = generate_alias_candidates(names)

    print(f"Found {len(alias_candidates)} alias candidates.")

    print("\nGenerating semantic-neighbour candidates...")

    semantic_candidates = generate_semantic_candidates(
        names,
        vectors
    )

    print(
        f"Found {len(semantic_candidates)} semantic candidates."
    )

    selected = select_candidates(
        alias_candidates,
        semantic_candidates,
        existing_pairs
    )

    # Calculate similarities for all selected pairs.
    model = SentenceTransformer(MODEL_NAME)

    alias_similarities = calculate_alias_similarities(
        alias_candidates,
        lookup,
        model
    )

    results = []

    for pair, information in selected.items():

        a, b = pair

        if pair in alias_similarities:

            similarity = alias_similarities[pair][
                "cosine_similarity"
            ]

        else:

            similarity = float(np.dot(lookup[a], lookup[b]))

        results.append({
            "ingredient_a": a,
            "ingredient_b": b,
            "cosine_similarity": similarity,
            "candidate_source": information["candidate_source"],
            "label": ""
        })

    output_df = pd.DataFrame(results)

    output_df = output_df.sort_values(
        by=["candidate_source", "cosine_similarity"],
        ascending=[True, False]
    )

    output_df.to_csv(OUTPUT_FILE, index=False)

    print("\nCANDIDATE GENERATION COMPLETE")
    print("-" * 45)

    print(f"New candidate pairs: {len(output_df)}")

    print(
        "Parenthetical candidates:",
        (output_df["candidate_source"] ==
         "Parenthetical name candidate").sum()
    )

    print(
        "Semantic candidates:",
        (output_df["candidate_source"] ==
         "Semantic neighbour candidate").sum()
    )

    print(f"\nSaved to:\n{OUTPUT_FILE}")

    print("\nFirst 20 candidates:")

    print(
        output_df.head(20).to_string(index=False)
    )


if __name__ == "__main__":
    main()


In [ ]:
#test ingredient similarity

import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Define project paths
BASE_DIR = Path(__file__).resolve().parent.parent

EMBEDDINGS_PATH = (
    BASE_DIR / "data" / "processed" / "ingredient_embeddings.npz"
)

# Load ingredients and embeddings
data = np.load(EMBEDDINGS_PATH)

ingredients = data["ingredients"].tolist()
embeddings = data["embeddings"]

# Create ingredient-to-index mapping
ingredient_index = {
    ingredient: index
    for index, ingredient in enumerate(ingredients)
}

# Load model for ingredients not in the vocabulary
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_embedding(ingredient):
    ingredient = ingredient.strip().lower()

    if ingredient in ingredient_index:
        return embeddings[ingredient_index[ingredient]]

    return model.encode(
        ingredient,
        convert_to_numpy=True,
        normalize_embeddings=True
    )


def compare_ingredients(ingredient_a, ingredient_b):
    embedding_a = get_embedding(ingredient_a)
    embedding_b = get_embedding(ingredient_b)

    similarity = np.dot(embedding_a, embedding_b)

    print(
        f"{ingredient_a} <-> {ingredient_b}: "
        f"{similarity:.4f}"
    )



test_pairs = [
    ("carrot", "carrots (gajjar)"),
    ("cow peas", "black eyed peas (lobia)"),
    ("aubergine", "eggplant"),
    ("minced beef", "ground beef"),
    ("milk", "coconut milk"),
    ("cow milk", "milk"),
    ("potato", "sweet potato"),
    ("green peas", "black eyed peas")
]

print("\nSimilarity results\n")

for ingredient_a, ingredient_b in test_pairs:
    compare_ingredients(ingredient_a, ingredient_b)


In [ ]:
# generate_ingredient_pair features

import numpy as np
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer




BASE_DIR = Path(__file__).resolve().parent.parent

TEST_DATA_PATH = Path("data/processed/ingredient_matching_tests.csv")

EMBEDDINGS_PATH = Path ("data/processed/ingredient_embeddings.npz")

OUTPUT_PATH = Path("data/processed/ingredient_pair_features.csv")


# 2. Load ingredient pair data
if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Test dataset not found: {TEST_DATA_PATH}"
    )

pairs_df = pd.read_csv(TEST_DATA_PATH)

required_columns = ["ingredient_a", "ingredient_b", "label"]

if not all(column in pairs_df.columns for column in required_columns):
    raise ValueError(
        f"Dataset must contain these columns: {required_columns}"
    )

if pairs_df[required_columns].isnull().any().any():
    raise ValueError("The ingredient pair dataset contains missing values.")

if not pairs_df["label"].isin([0, 1]).all():
    raise ValueError("Labels must contain only 0 or 1.")



# 3. Load existing ingredient embeddings
if not EMBEDDINGS_PATH.exists():
    raise FileNotFoundError(
        f"Embedding file not found: {EMBEDDINGS_PATH}"
    )

embedding_data = np.load(EMBEDDINGS_PATH)

ingredients = embedding_data["ingredients"].tolist()
embeddings = embedding_data["embeddings"]

# Map each ingredient to its embedding
ingredient_index = {
    ingredient: index
    for index, ingredient in enumerate(ingredients)
}

print(f"Loaded {len(ingredients)} ingredient embeddings.")



# 4. Load model 

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Cache embeddings generated for unseen ingredients
embedding_cache = {}


def get_embedding(ingredient):
    """
    Retrieve an existing embedding or generate one
    if the ingredient is not in the vocabulary.
    """

    ingredient = str(ingredient).strip().lower()

    if ingredient in ingredient_index:
        return embeddings[ingredient_index[ingredient]]

    if ingredient not in embedding_cache:
        embedding_cache[ingredient] = model.encode(
            ingredient,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    return embedding_cache[ingredient]



# 5. Calculate ingredient pair features
def calculate_features(ingredient_a, ingredient_b):

    ingredient_a = str(ingredient_a).strip().lower()
    ingredient_b = str(ingredient_b).strip().lower()

    embedding_a = get_embedding(ingredient_a)
    embedding_b = get_embedding(ingredient_b)

    # Embeddings are normalized, so their dot product
    # gives cosine similarity.
    cosine_similarity = float(
        np.dot(embedding_a, embedding_b)
    )

    # Calculate embedding distance
    embedding_distance = 1.0 - cosine_similarity

    # Split ingredient names into words
    words_a = set(ingredient_a.split())
    words_b = set(ingredient_b.split())

    # Calculate word overlap using Jaccard similarity
    union = words_a | words_b
    intersection = words_a & words_b

    word_overlap = (
        len(intersection) / len(union)
        if union else 0.0
    )

    # Calculate difference in ingredient name lengths
    length_difference = abs(
        len(ingredient_a) - len(ingredient_b)
    )

    # Calculate relative length difference
    maximum_length = max(
        len(ingredient_a),
        len(ingredient_b),
        1
    )

    normalized_length_difference = (
        length_difference / maximum_length
    )

    return {
        "cosine_similarity": cosine_similarity,
        "embedding_distance": embedding_distance,
        "word_overlap": word_overlap,
        "length_difference": length_difference,
        "normalized_length_difference": normalized_length_difference
    }



# 6. Generate features for every ingredient pair
feature_records = []

for _, row in pairs_df.iterrows():

    ingredient_a = row["ingredient_a"]
    ingredient_b = row["ingredient_b"]

    features = calculate_features(
        ingredient_a,
        ingredient_b
    )

    feature_records.append({
        "ingredient_a": ingredient_a,
        "ingredient_b": ingredient_b,
        **features,
        "label": int(row["label"])
    })



features_df = pd.DataFrame(feature_records)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

features_df.to_csv(OUTPUT_PATH, index=False)

print("\nFeature generation completed")

print(f"Ingredient pairs processed {len(features_df)}")
print(f"Equivalent pairs {(features_df['label'] == 1).sum()}")
print(f"Non-equivalent pairs {(features_df['label'] == 0).sum()}")

print(f"\nSaved features to{OUTPUT_PATH}")

print("\nGenerated features")
print(features_df.to_string(index=False))


In [ ]:
#auto label ingredient pairs

import re
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. File paths and settings
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent

INPUT_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_pair_review.csv"
)

MANUAL_LABELS_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_matching_tests.csv"
)

EMBEDDINGS_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_embeddings.npz"
)

OUTPUT_FILE = (
    BASE_DIR / "data" / "processed" /
    "ingredient_pair_auto_labels.csv"
)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

POSITIVE_SIMILARITY = 0.85



def normalize_name(name):
    name = str(name).lower().strip()
    name = re.sub(r"\s+", " ", name)
    return name


def get_tokens(name):
    return set(re.findall(r"\b\w+\b", normalize_name(name)))



def load_embeddings():
    data = np.load(EMBEDDINGS_FILE)

    names = data["ingredients"]
    vectors = data["embeddings"]

    return {
        normalize_name(name): vector
        for name, vector in zip(names, vectors)
    }




def calculate_features(name_a, name_b, embedding_lookup, model):

    name_a = normalize_name(name_a)
    name_b = normalize_name(name_b)

    tokens_a = get_tokens(name_a)
    tokens_b = get_tokens(name_b)

    intersection = tokens_a.intersection(tokens_b)
    union = tokens_a.union(tokens_b)

    word_overlap = len(intersection) / len(union) if union else 0

    smaller_count = min(len(tokens_a), len(tokens_b))

    token_containment = (
        len(intersection) / smaller_count
        if smaller_count else 0
    )

    character_similarity = SequenceMatcher(
        None, name_a, name_b
    ).ratio()

    length_difference = abs(len(name_a) - len(name_b))

    longest_length = max(len(name_a), len(name_b), 1)

    normalized_length_difference = (
        length_difference / longest_length
    )

    missing = [
        name for name in (name_a, name_b)
        if name not in embedding_lookup
    ]

    if missing:
        unique_missing = list(dict.fromkeys(missing))

        vectors = model.encode(
            unique_missing,
            normalize_embeddings=True
        )

        for name, vector in zip(unique_missing, vectors):
            embedding_lookup[name] = vector

    vector_a = embedding_lookup[name_a]
    vector_b = embedding_lookup[name_b]

    cosine_similarity = float(np.dot(vector_a, vector_b))

    return {
        "cosine_similarity": cosine_similarity,
        "embedding_distance": 1 - cosine_similarity,
        "word_overlap": word_overlap,
        "token_containment": token_containment,
        "character_similarity": character_similarity,
        "length_difference": length_difference,
        "normalized_length_difference": normalized_length_difference
    }


# --------------------------------------------------
# 5. Conservative automatic labelling
# --------------------------------------------------

def assign_label(name_a, name_b, features):

    name_a = normalize_name(name_a)
    name_b = normalize_name(name_b)

    similarity = features["cosine_similarity"]
    tokens_a = get_tokens(name_a)
    tokens_b = get_tokens(name_b)

    if name_a == name_b:
        return 1, "Identical ingredient names"

    # Explicit modifiers can distinguish ingredient types.
    distinguishing_modifiers = {
        "purple", "sweet", "coconut", "brown",
        "white", "black", "red", "green",
        "yellow", "ground", "whole", "dried",
        "fresh", "smoked", "powdered"
    }

    shared_tokens = tokens_a.intersection(tokens_b)

    modifiers_a = tokens_a.intersection(distinguishing_modifiers)
    modifiers_b = tokens_b.intersection(distinguishing_modifiers)

    if modifiers_a != modifiers_b and (
        modifiers_a or modifiers_b
    ):
        return 0, "Different ingredient modifiers"

    # High similarity and matching wording.
    if (
        similarity >= POSITIVE_SIMILARITY
        and features["token_containment"] >= 0.80
    ):
        return 1, "High semantic similarity and token containment"

    # Very low similarity is not sufficient to reject a pair.
    # Unknown pairs remain REVIEW.
    return "REVIEW", "Requires further verification"



def load_manual_labels():

    manual_df = pd.read_csv(MANUAL_LABELS_FILE)

    manual_df["ingredient_a"] = (
        manual_df["ingredient_a"].apply(normalize_name)
    )

    manual_df["ingredient_b"] = (
        manual_df["ingredient_b"].apply(normalize_name)
    )

    manual_labels = {}

    for _, row in manual_df.iterrows():

        pair = tuple(sorted([
            row["ingredient_a"],
            row["ingredient_b"]
        ]))

        manual_labels[pair] = int(row["label"])

    return manual_labels



def main():

    df = pd.read_csv(INPUT_FILE)

    embedding_lookup = load_embeddings()

    manual_labels = load_manual_labels()

    model = SentenceTransformer(MODEL_NAME)

    results = []

    for _, row in df.iterrows():

        name_a = normalize_name(row["ingredient_a"])
        name_b = normalize_name(row["ingredient_b"])

        features = calculate_features(
            name_a,
            name_b,
            embedding_lookup,
            model
        )

        auto_label, reason = assign_label(
            name_a,
            name_b,
            features
        )

        pair = tuple(sorted([name_a, name_b]))

        manual_label = manual_labels.get(pair)

        final_label = (
            manual_label
            if manual_label is not None
            else auto_label
        )

        results.append({
            "ingredient_a": name_a,
            "ingredient_b": name_b,
            **features,
            "auto_label": auto_label,
            "manual_label": (
                manual_label if manual_label is not None else ""
            ),
            "final_label": final_label,
            "reason": (
                "Trusted manual label"
                if manual_label is not None
                else reason
            )
        })

    output_df = pd.DataFrame(results)

    output_df.to_csv(OUTPUT_FILE, index=False)

    print("\nINGREDIENT LABELLING RESULTS")
    print("-" * 45)

    print(f"Total pairs: {len(output_df)}")

    print(
        "Manually labelled:",
        output_df["manual_label"].ne("").sum()
    )

    print(
        "Automatically equivalent:",
        (output_df["auto_label"] == 1).sum()
    )

    print(
        "Automatically different:",
        (output_df["auto_label"] == 0).sum()
    )

    print(
        "Needs review:",
        (output_df["final_label"] == "REVIEW").sum()
    )

    print(f"\nSaved results to:\n{OUTPUT_FILE}")

    print("\nFirst 15 results:")

    print(
        output_df[
            [
                "ingredient_a",
                "ingredient_b",
                "auto_label",
                "manual_label",
                "final_label"
            ]
        ].head(15).to_string(index=False)
    )


if __name__ == "__main__":
    main()


In [ ]:
#consolidating ingredient pair data

from pathlib import Path
import pandas as pd


# --------------------------------------------------
# 1. Project paths
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

OUTPUT_FILE = PROCESSED_DIR / "ingredient_pairs.csv"
INPUT_FILES = {
    "manual": PROCESSED_DIR / "ingredient_matching_tests.csv",
    "review": PROCESSED_DIR / "ingredient_pair_review.csv",
    "candidates": PROCESSED_DIR / "ingredient_pair_candidates_200.csv",
    "features": PROCESSED_DIR / "ingredient_pair_features.csv",
    "automatic": PROCESSED_DIR / "ingredient_pair_auto_labels.csv",
}


FEATURE_COLUMNS = [
    "cosine_similarity",
    "embedding_distance",
    "word_overlap",
    "token_containment",
    "character_similarity",
    "length_difference",
    "normalized_length_difference",
]



def normalize_name(value):
    """Standardize ingredient names."""
    if pd.isna(value):
        return ""

    return " ".join(str(value).lower().strip().split())


def normalize_label(value):
    """Accept only verified binary labels."""
    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value in {"1", "1.0"}:
        return 1

    if value in {"0", "0.0"}:
        return 0

    return None


def make_pair_key(name_a, name_b):
    """Create an order-independent pair identifier."""
    return tuple(sorted((name_a, name_b)))


def safe_float(value):
    """Convert a feature to a number where possible."""
    if pd.isna(value):
        return None

    try:
        return float(value)
    except (ValueError, TypeError):
        return None


# --------------------------------------------------
# 3. Read existing CSV files
# --------------------------------------------------

def load_input_files():

    loaded_files = {}

    for source, file_path in INPUT_FILES.items():

        if not file_path.exists():
            print(f"Skipping missing file: {file_path.name}")
            continue

        df = pd.read_csv(file_path)

        required = {"ingredient_a", "ingredient_b"}

        if not required.issubset(df.columns):
            print(f"Skipping invalid file: {file_path.name}")
            continue

        loaded_files[source] = df

        print(
            f"Loaded {file_path.name}: {len(df)} rows"
        )

    if not loaded_files:
        raise FileNotFoundError(
            "No valid ingredient-pair CSV files were found."
        )

    return loaded_files


# --------------------------------------------------
# 4. Determine which labels are trusted
# --------------------------------------------------

def get_trusted_label(row, source):
    """
    Only use human-provided labels.

    Never use auto_label or final_label as verified labels.
    """

    if source == "automatic":

        # The automatic file has a separate manual_label column.
        if "manual_label" in row.index:
            return normalize_label(row["manual_label"])

        return None

    # Labels in the manual, review, candidate and feature
    # files are accepted as supplied human labels.
    if "label" in row.index:
        return normalize_label(row["label"])

    return None


# --------------------------------------------------
# 5. Consolidate all ingredient pairs
# --------------------------------------------------

def consolidate_pairs(loaded_files):

    consolidated = {}

    # Priority determines which source supplies the first
    # available feature value and preferred auto prediction.
    priority = [
        "manual",
        "review",
        "candidates",
        "features",
        "automatic",
    ]

    for source in priority:

        if source not in loaded_files:
            continue

        df = loaded_files[source]

        for _, row in df.iterrows():

            name_a = normalize_name(row["ingredient_a"])
            name_b = normalize_name(row["ingredient_b"])

            if not name_a or not name_b:
                continue

            if name_a == name_b:
                continue

            pair_key = make_pair_key(name_a, name_b)

            # Keep a consistent ingredient order.
            name_a, name_b = pair_key

            if pair_key not in consolidated:

                consolidated[pair_key] = {
                    "ingredient_a": name_a,
                    "ingredient_b": name_b,
                    "label": None,
                    "auto_label": "REVIEW",
                    "status": "PENDING",
                    "source": set(),
                    "reason": "",
                    "conflict": False,
                }

                for feature in FEATURE_COLUMNS:
                    consolidated[pair_key][feature] = None

            record = consolidated[pair_key]

            record["source"].add(source)

            # Preserve the first available feature value.
            for feature in FEATURE_COLUMNS:

                if feature in row.index:

                    value = safe_float(row[feature])

                    if (
                        value is not None
                        and record[feature] is None
                    ):
                        record[feature] = value

            # Preserve automatic predictions separately.
            if source == "automatic" and "auto_label" in row.index:

                prediction = str(row["auto_label"]).strip().upper()

                if prediction in {"0", "0.0"}:
                    record["auto_label"] = 0

                elif prediction in {"1", "1.0"}:
                    record["auto_label"] = 1

                elif prediction == "REVIEW":
                    record["auto_label"] = "REVIEW"

            # Preserve a useful explanation.
            if "reason" in row.index and pd.notna(row["reason"]):

                reason = str(row["reason"]).strip()

                if reason and not record["reason"]:
                    record["reason"] = reason

            # Only trusted labels are considered here.
            trusted_label = get_trusted_label(row, source)

            if trusted_label is not None:

                if record["label"] is None:

                    record["label"] = trusted_label

                elif record["label"] != trusted_label:

                    record["conflict"] = True

    return consolidated


# --------------------------------------------------
# 6. Build the final DataFrame
# --------------------------------------------------

def build_output(consolidated):

    results = []

    for record in consolidated.values():

        if record["conflict"]:

            status = "CONFLICT"

            # Do not silently choose between conflicting labels.
            label = None

        elif record["label"] is not None:

            status = "REVIEWED"
            label = record["label"]

        else:

            status = "PENDING"
            label = None

        results.append({
            "ingredient_a": record["ingredient_a"],
            "ingredient_b": record["ingredient_b"],
            **{
                feature: record[feature]
                for feature in FEATURE_COLUMNS
            },
            "label": label if label is not None else "",
            "auto_label": record["auto_label"],
            "status": status,
            "source": ";".join(sorted(record["source"])),
            "reason": record["reason"],
        })

    output_df = pd.DataFrame(results)

    return output_df.sort_values(
        by=["status", "ingredient_a", "ingredient_b"]
    ).reset_index(drop=True)


# --------------------------------------------------
# 7. Main program
# --------------------------------------------------

def main():

    print("\nINGREDIENT PAIR MIGRATION")
    print("=" * 45)

    loaded_files = load_input_files()

    consolidated = consolidate_pairs(loaded_files)

    output_df = build_output(consolidated)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    # Prevent accidental overwriting of an existing central dataset.
    if OUTPUT_FILE.exists():

        raise FileExistsError(
            f"\n{OUTPUT_FILE.name} already exists.\n"
            "Back it up or rename it before running this migration."
        )

    output_df.to_csv(OUTPUT_FILE, index=False)

    print("\nMIGRATION COMPLETE")
    print("=" * 45)

    print(f"Total unique pairs: {len(output_df)}")

    print(
        "Reviewed pairs:",
        (output_df["status"] == "REVIEWED").sum()
    )

    print(
        "Pending pairs:",
        (output_df["status"] == "PENDING").sum()
    )

    print(
        "Conflicting pairs:",
        (output_df["status"] == "CONFLICT").sum()
    )

    print(f"\nOutput saved to:\n{OUTPUT_FILE}")

    print("\nFirst 15 consolidated pairs:")

    print(
        output_df[
            [
                "ingredient_a",
                "ingredient_b",
                "label",
                "auto_label",
                "status",
                "source",
            ]
        ].head(15).to_string(index=False)
    )


if __name__ == "__main__":
    main()


In [ ]:
# train ingredient-equivalence classifier

from pathlib import Path
from difflib import SequenceMatcher
import json
import shutil

import joblib
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)




BASE_DIR = Path(__file__).resolve().parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "ingredient_matching"

CSV_FILE = PROCESSED_DIR / "ingredient_pairs.csv"
EMBEDDINGS_FILE = PROCESSED_DIR / "ingredient_embeddings.npz"

MODEL_FILE = MODEL_DIR / "ingredient_equivalence_model.joblib"
METRICS_FILE = MODEL_DIR / "evaluation_metrics.json"

BACKUP_FILE = PROCESSED_DIR / "ingredient_pairs_before_classifier.bak"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CONFIDENCE_THRESHOLD = 0.75
TEST_SIZE = 0.20
RANDOM_STATE = 42

FEATURE_COLUMNS = [
    "cosine_similarity",
    "embedding_distance",
    "word_overlap",
    "token_containment",
    "character_similarity",
    "length_difference",
    "normalized_length_difference",
]


# ==================================================
# 2. NORMALIZATION AND FEATURE FUNCTIONS
# ==================================================

def normalize_name(value):
    if pd.isna(value):
        return ""

    return " ".join(str(value).lower().strip().split())


def normalize_label(value):
    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value in {"1", "1.0"}:
        return 1

    if value in {"0", "0.0"}:
        return 0

    return None


def calculate_text_features(name_a, name_b):

    tokens_a = set(name_a.split())
    tokens_b = set(name_b.split())

    union = tokens_a | tokens_b
    intersection = tokens_a & tokens_b

    word_overlap = (
        len(intersection) / len(union)
        if union else 0
    )

    token_containment = (
        len(intersection) / min(len(tokens_a), len(tokens_b))
        if tokens_a and tokens_b else 0
    )

    length_difference = abs(len(name_a) - len(name_b))

    maximum_length = max(len(name_a), len(name_b), 1)

    normalized_length_difference = (
        length_difference / maximum_length
    )

    character_similarity = SequenceMatcher(
        None, name_a, name_b
    ).ratio()

    return {
        "word_overlap": word_overlap,
        "token_containment": token_containment,
        "character_similarity": character_similarity,
        "length_difference": length_difference,
        "normalized_length_difference": normalized_length_difference,
    }


# ==================================================
# 3. LOAD DATA AND PRESERVE ORIGINAL LABELS
# ==================================================

def load_data():

    if not CSV_FILE.exists():
        raise FileNotFoundError(f"Missing dataset: {CSV_FILE}")

    if not EMBEDDINGS_FILE.exists():
        raise FileNotFoundError(
            f"Missing embeddings: {EMBEDDINGS_FILE}"
        )

    df = pd.read_csv(CSV_FILE)

    required_columns = {"ingredient_a", "ingredient_b"}

    if not required_columns.issubset(df.columns):
        raise ValueError("The CSV is missing ingredient name columns.")

    # Preserve the existing CSV before modifying it.
    if not BACKUP_FILE.exists():
        shutil.copy2(CSV_FILE, BACKUP_FILE)
        print(f"Backup created: {BACKUP_FILE}")

    df["ingredient_a"] = df["ingredient_a"].apply(normalize_name)
    df["ingredient_b"] = df["ingredient_b"].apply(normalize_name)

    # Preserve the existing manually assigned labels.
    if "manual_label" not in df.columns:

        df["manual_label"] = df["label"].apply(normalize_label)

    else:

        df["manual_label"] = df["manual_label"].apply(normalize_label)

    # Do not use automatic labels as training targets.
    df["label"] = df["manual_label"]

    return df


# ==================================================
# 4. LOAD EMBEDDINGS AND GENERATE MISSING ONES
# ==================================================

def load_embeddings(df):

    saved = np.load(EMBEDDINGS_FILE, allow_pickle=True)

    vocabulary = [
        normalize_name(name)
        for name in saved["ingredients"]
    ]

    vectors = saved["embeddings"]

    embedding_lookup = {
        name: vector
        for name, vector in zip(vocabulary, vectors)
    }

    required_names = set(df["ingredient_a"]) | set(df["ingredient_b"])

    missing_names = sorted(
        name for name in required_names
        if name not in embedding_lookup
    )

    if missing_names:

        print(
            f"Generating embeddings for {len(missing_names)} "
            "previously unseen ingredient names..."
        )

        model = SentenceTransformer(EMBEDDING_MODEL_NAME)

        new_vectors = model.encode(
            missing_names,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=64,
        )

        for name, vector in zip(missing_names, new_vectors):
            embedding_lookup[name] = vector

    return embedding_lookup


# ==================================================
# 5. CALCULATE FEATURES FOR EVERY PAIR
# ==================================================

def calculate_all_features(df, embedding_lookup):

    feature_rows = []

    for _, row in df.iterrows():

        name_a = row["ingredient_a"]
        name_b = row["ingredient_b"]

        vector_a = embedding_lookup[name_a]
        vector_b = embedding_lookup[name_b]

        cosine_similarity = float(
            np.dot(vector_a, vector_b)
        )

        features = calculate_text_features(name_a, name_b)

        features["cosine_similarity"] = cosine_similarity
        features["embedding_distance"] = 1 - cosine_similarity

        feature_rows.append(features)

    feature_df = pd.DataFrame(feature_rows, index=df.index)

    for column in FEATURE_COLUMNS:
        df[column] = feature_df[column]

    return df


# ==================================================
# 6. PREPARE TRAINING DATA
# ==================================================

def prepare_training_data(df):

    labelled = df[df["manual_label"].notna()].copy()

    if len(labelled) < 20:
        raise ValueError(
            "At least 20 manually labelled pairs are required."
        )

    X = labelled[FEATURE_COLUMNS].astype(float)

    y = labelled["manual_label"].astype(int)

    class_counts = y.value_counts()

    print("\nTRAINING LABEL DISTRIBUTION")
    print(class_counts.to_string())

    if len(class_counts) != 2 or class_counts.min() < 2:
        raise ValueError(
            "Both classes need at least two labelled examples."
        )

    return X, y


# ==================================================
# 7. CREATE AND TRAIN THE CLASSIFIER
# ==================================================

def create_classifier():

    return Pipeline([
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ])



def evaluate_classifier(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    evaluation_model = create_classifier()

    evaluation_model.fit(X_train, y_train)

    predictions = evaluation_model.predict(X_test)

    probabilities = evaluation_model.predict_proba(X_test)[:, 1]

    metrics = {
        "training_samples": int(len(X_train)),
        "testing_samples": int(len(X_test)),
        "accuracy": float(accuracy_score(y_test, predictions)),
        "precision": float(
            precision_score(y_test, predictions, zero_division=0)
        ),
        "recall": float(
            recall_score(y_test, predictions, zero_division=0)
        ),
        "f1_score": float(
            f1_score(y_test, predictions, zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_test, predictions
        ).tolist(),
        "classification_report": classification_report(
            y_test,
            predictions,
            zero_division=0,
        ),
    }

    if len(np.unique(y_test)) == 2:
        metrics["roc_auc"] = float(
            roc_auc_score(y_test, probabilities)
        )

    print("\nHELD-OUT EVALUATION")
    print("=" * 45)

    for metric in [
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
    ]:
        if metric in metrics:
            print(f"{metric}: {metrics[metric]:.4f}")

    print("\nConfusion matrix:")
    print(metrics["confusion_matrix"])

    print("\nClassification report:")
    print(metrics["classification_report"])

    return metrics




def train_final_model(X, y):

    model = create_classifier()

    model.fit(X, y)

    return model


#ingredient pairing
def label_all_pairs(df, model):

    X_all = df[FEATURE_COLUMNS].astype(float)

    probabilities = model.predict_proba(X_all)

    predicted_classes = model.predict(X_all)

    confidence_values = probabilities.max(axis=1)

    raw_predictions = []

    for prediction, confidence in zip(
        predicted_classes,
        confidence_values,
    ):

        if confidence < CONFIDENCE_THRESHOLD:
            raw_predictions.append("REVIEW")

        else:
            raw_predictions.append(int(prediction))

    df["auto_label"] = raw_predictions

    df["confidence"] = confidence_values

    # Preserve the original manual labels.
    # For unlabelled pairs, convert REVIEW to 0 as requested.
    final_labels = []
    statuses = []

    for _, row in df.iterrows():

        manual_label = row["manual_label"]
        automatic_label = row["auto_label"]

        if pd.notna(manual_label):

            final_labels.append(int(manual_label))
            statuses.append("MANUAL_SEED")

        elif automatic_label == "REVIEW":

            final_labels.append(0)
            statuses.append("REVIEW_CONVERTED_TO_0")

        else:

            final_labels.append(int(automatic_label))
            statuses.append("AUTO_ASSIGNED")

    df["label"] = final_labels
    df["status"] = statuses

    return df



def save_results(df, model, metrics):

    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, MODEL_FILE)

    with open(METRICS_FILE, "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=4)

    df.to_csv(CSV_FILE, index=False)

    print("\nSAVED RESULTS")
    print("=" * 45)

    print(f"Updated dataset: {CSV_FILE}")
    print(f"Trained model: {MODEL_FILE}")
    print(f"Evaluation metrics: {METRICS_FILE}")
    print(f"Original CSV backup: {BACKUP_FILE}")

    print("\nFINAL LABEL COUNTS")
    print(df["label"].value_counts().to_string())

    print("\nAUTOMATIC LABELLING COUNTS")
    print(df["status"].value_counts().to_string())




def main():

    print("\nINGREDIENT EQUIVALENCE CLASSIFIER")
    print("=" * 45)

    df = load_data()

    print(f"\nLoaded ingredient pairs: {len(df)}")

    embedding_lookup = load_embeddings(df)

    df = calculate_all_features(df, embedding_lookup)

    X, y = prepare_training_data(df)

    metrics = evaluate_classifier(X, y)

    print("\nTraining final classifier using all manual labels...")

    final_model = train_final_model(X, y)

    df = label_all_pairs(df, final_model)

    save_results(df, final_model, metrics)

    print("\nCLASSIFIER TRAINING COMPLETED.")


if __name__ == "__main__":
    main()


In [ ]:
#Recommend using classifier model
import ast
import re
import sys
from pathlib import Path
from difflib import SequenceMatcher

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. PROJECT PATHS AND CONFIGURATION
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent

DATA_PATH = BASE_DIR / "data" / "processed" / "Recipes_cleaned.csv"

MODEL_PATH = (
    BASE_DIR
    / "models"
    / "ingredient_matching"
    / "ingredient_equivalence_model.joblib"
)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CONFIDENCE_THRESHOLD = 0.75
TOP_RESULTS = 10


# --------------------------------------------------
# 2. LOAD MODEL AND RECIPE DATA
# --------------------------------------------------

def load_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Ingredient classifier not found: {MODEL_PATH}"
        )

    print("Loading ingredient equivalence classifier...")
    return joblib.load(MODEL_PATH)


def load_recipes():
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Recipe dataset not found: {DATA_PATH}"
        )

    print("Loading recipes...")
    recipes = pd.read_csv(DATA_PATH)

    if "ingredients_normalized" not in recipes.columns:
        raise ValueError(
            "The dataset does not contain 'ingredients_normalized'."
        )

    name_column = next(
        (
            column for column in
            ["name", "recipe_name", "title", "recipe_title"]
            if column in recipes.columns
        ),
        None
    )

    if name_column is None:
        raise ValueError(
            "Could not identify the recipe name column. "
            "Check the columns in Recipes_cleaned.csv."
        )

    return recipes, name_column


# --------------------------------------------------
# 3. LOAD EMBEDDING MODEL
# --------------------------------------------------

def load_embedding_model():
    print("Loading ingredient embedding model...")
    return SentenceTransformer(EMBEDDING_MODEL_NAME)


# --------------------------------------------------
# 4. PROCESS INGREDIENTS
# --------------------------------------------------

def normalize_ingredient(ingredient):
    ingredient = str(ingredient).strip().lower()
    ingredient = re.sub(r"\s+", " ", ingredient)
    return ingredient


def parse_ingredients(value):
    """
    Convert the ingredients stored in the CSV into a list.
    Supports Python-style lists and comma-separated strings.
    """

    if pd.isna(value):
        return []

    if isinstance(value, list):
        ingredients = value

    else:
        value = str(value).strip()

        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                ingredients = parsed
            else:
                ingredients = value.split(",")

        except (ValueError, SyntaxError):
            ingredients = value.split(",")

    cleaned = []

    for ingredient in ingredients:
        ingredient = normalize_ingredient(ingredient)

        if ingredient and ingredient not in cleaned:
            cleaned.append(ingredient)

    return cleaned


def get_user_ingredients():
    print("\nEnter ingredients separated by commas.")

    user_input = input("Your ingredients: ").strip()

    ingredients = [
        normalize_ingredient(item)
        for item in user_input.split(",")
    ]

    ingredients = list(dict.fromkeys(
        ingredient for ingredient in ingredients if ingredient
    ))

    if not ingredients:
        raise ValueError("You must enter at least one ingredient.")

    return ingredients


# --------------------------------------------------
# 5. GENERATE CLASSIFIER FEATURES
# --------------------------------------------------

def calculate_features(ingredient_a, ingredient_b, embedding_cache):

    ingredient_a = normalize_ingredient(ingredient_a)
    ingredient_b = normalize_ingredient(ingredient_b)

    embedding_a = embedding_cache[ingredient_a]
    embedding_b = embedding_cache[ingredient_b]

    cosine_similarity = float(
        np.dot(embedding_a, embedding_b)
    )

    embedding_distance = 1.0 - cosine_similarity

    words_a = set(ingredient_a.split())
    words_b = set(ingredient_b.split())

    intersection = words_a & words_b
    union = words_a | words_b

    word_overlap = (
        len(intersection) / len(union)
        if union else 0.0
    )

    token_containment = (
        len(intersection) / min(len(words_a), len(words_b))
        if words_a and words_b else 0.0
    )

    character_similarity = SequenceMatcher(
        None,
        ingredient_a,
        ingredient_b
    ).ratio()

    length_difference = abs(
        len(ingredient_a) - len(ingredient_b)
    )

    maximum_length = max(
        len(ingredient_a),
        len(ingredient_b),
        1
    )

    normalized_length_difference = (
        length_difference / maximum_length
    )

    return [
        cosine_similarity,
        embedding_distance,
        word_overlap,
        token_containment,
        character_similarity,
        length_difference,
        normalized_length_difference
    ]


# --------------------------------------------------
# 6. CREATE INGREDIENT EMBEDDINGS
# --------------------------------------------------

def create_embedding_cache(ingredients, embedding_model):

    unique_ingredients = list(dict.fromkeys(ingredients))

    embeddings = embedding_model.encode(
        unique_ingredients,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    return {
        ingredient: embedding
        for ingredient, embedding in zip(
            unique_ingredients,
            embeddings
        )
    }


# --------------------------------------------------
# 7. PREDICT INGREDIENT EQUIVALENCE
# --------------------------------------------------

def predict_equivalence(
    ingredient_a,
    ingredient_b,
    classifier,
    embedding_cache
):

    feature_names = [
        "cosine_similarity",
        "embedding_distance",
        "word_overlap",
        "token_containment",
        "character_similarity",
        "length_difference",
        "normalized_length_difference"
    ]

    features = calculate_features(
        ingredient_a,
        ingredient_b,
        embedding_cache
    )

    features_df = pd.DataFrame(
        [features],
        columns=feature_names
    )

    probabilities = classifier.predict_proba(features_df)[0]

    positive_class_index = list(
        classifier.classes_
    ).index(1)

    return float(probabilities[positive_class_index])

# --------------------------------------------------
# 8. MATCH USER INGREDIENTS TO A RECIPE
# --------------------------------------------------

def match_recipe(
    user_ingredients,
    recipe_ingredients,
    classifier,
    embedding_cache
):

    matched = []
    missing = []

    for user_ingredient in user_ingredients:

        best_match = None
        best_score = 0.0

        for recipe_ingredient in recipe_ingredients:

            score = predict_equivalence(
                user_ingredient,
                recipe_ingredient,
                classifier,
                embedding_cache
            )

            if score > best_score:
                best_score = score
                best_match = recipe_ingredient

        if best_match is not None and best_score >= CONFIDENCE_THRESHOLD:

            matched.append({
                "user_ingredient": user_ingredient,
                "recipe_ingredient": best_match,
                "confidence": best_score
            })

        else:
            missing.append(user_ingredient)

    return matched, missing


# --------------------------------------------------
# 9. RANK RECIPES
# --------------------------------------------------

def recommend_recipes(
    user_ingredients,
    recipes,
    name_column,
    classifier,
    embedding_model
):

    print("\nPreparing ingredient comparisons...")

    parsed_recipes = []

    all_ingredients = list(user_ingredients)

    for _, row in recipes.iterrows():

        recipe_ingredients = parse_ingredients(
            row["ingredients_normalized"]
        )

        if not recipe_ingredients:
            continue

        parsed_recipes.append({
            "name": row[name_column],
            "ingredients": recipe_ingredients
        })

        all_ingredients.extend(recipe_ingredients)

    embedding_cache = create_embedding_cache(
        all_ingredients,
        embedding_model
    )

    print("Comparing ingredients with recipes...")

    results = []

    for recipe in parsed_recipes:

        matched, missing = match_recipe(
            user_ingredients,
            recipe["ingredients"],
            classifier,
            embedding_cache
        )

        matched_count = len(matched)

        score = (
            matched_count / len(user_ingredients)
        ) * 100

        results.append({
            "name": recipe["name"],
            "ingredients": recipe["ingredients"],
            "matched": matched,
            "missing": missing,
            "score": score,
            "matched_count": matched_count
        })

    results.sort(
        key=lambda item: (
            item["score"],
            item["matched_count"]
        ),
        reverse=True
    )

    return results[:TOP_RESULTS]




def display_recommendations(results, user_ingredients):

    print("\n" + "=" * 65)
    print("           INGREDIENT-BASED RECIPE RECOMMENDATIONS")
    print("=" * 65)

    print("\nYour ingredients:")
    print(", ".join(user_ingredients))

    if not results:
        print("\nNo suitable recipes were found.")
        return

    for index, recipe in enumerate(results, start=1):

        print("\n" + "-" * 65)
        print(f"{index}. {recipe['name']}")

        print(
            f"Match score: {recipe['score']:.2f}% "
            f"({recipe['matched_count']}/{len(user_ingredients)} "
            f"ingredients matched)"
        )

        print("\nMatched ingredients:")

        if recipe["matched"]:
            for item in recipe["matched"]:
                print(
                    f"  {item['user_ingredient']} "
                    f"-> {item['recipe_ingredient']} "
                    f"({item['confidence']:.2%})"
                )
        else:
            print("  No ingredients matched.")

        print("\nIngredients you do not have a match for:")

        if recipe["missing"]:
            for ingredient in recipe["missing"]:
                print(f"  - {ingredient}")
        else:
            print("  None")

        print("\nRecipe ingredients:")
        print(", ".join(recipe["ingredients"]))


def main():

    try:

        classifier = load_model()

        recipes, name_column = load_recipes()

        embedding_model = load_embedding_model()

        user_ingredients = get_user_ingredients()

        results = recommend_recipes(
            user_ingredients,
            recipes,
            name_column,
            classifier,
            embedding_model
        )

        display_recommendations(
            results,
            user_ingredients
        )

    except Exception as error:

        print(f"\nError: {error}")
        sys.exit(1)


if __name__ == "__main__":
    main()